# Experiment 3.4 (directory retained): Clamped scalar norm-ratio rescaling for Muon

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/CURVATURE_SCALED_muon/run_experiment.py`  
**Notebook identity:** this notebook imports that script's `run_experiment()` backend and analyzes its returned results. It does **not** duplicate the training logic.

## Honest scope

The directory name `CURVATURE_SCALED_muon` is historical. The implemented method is **not** an explicit curvature or Hessian estimator. The main dynamic rule is a **clipped scalar Frobenius-norm ratio**

\[
\text{scale} = \mathrm{clip}\left(\|G\|_F / \|\mathrm{ortho}(G)\|_F \cdot \gamma,\; \text{scale\_min},\; \text{scale\_max}\right)
\]

applied after Newton--Schulz orthogonalization.

This notebook asks a narrower toy question:

1. Does plain `k=20` Muon underperform plain `k=5` on this deep-linear problem?
2. Does the clipped scalar norm-ratio rescaling improve `k=20` and `k=5`?
3. Does a simple fixed-scale control (`k=20 + fixed scale 0.1`) explain most of the improvement?
4. How often do the winning rescaled variants sit on the clamp boundary?

## Toy-study limits

- 2-layer `4x4` deep linear network
- `32` data points
- `500` training steps
- `10` seeds
- no explicit Hessian, curvature, or Newton-alignment measurement

So any conclusion here is about **empirical optimization behavior in a toy setting**, not a direct demonstration of curvature restoration.


In [ ]:
from pathlib import Path
import importlib.util
import platform
import time

import matplotlib.pyplot as plt
import numpy as np

for style_name in ['seaborn-v0_8-whitegrid', 'seaborn-whitegrid', 'default']:
    try:
        plt.style.use(style_name)
        break
    except OSError:
        continue

EXPERIMENT_RELATIVE_PATH = Path(
    'experiments/Tier_1_Core_Mechanism_Tests/CURVATURE_SCALED_muon/run_experiment.py'
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / EXPERIMENT_RELATIVE_PATH).exists():
            return candidate
    raise FileNotFoundError(
        f'Could not locate project root containing {EXPERIMENT_RELATIVE_PATH} from {start}'
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODULE_PATH = PROJECT_ROOT / EXPERIMENT_RELATIVE_PATH

spec = importlib.util.spec_from_file_location('curvature_scaled_muon_backend', MODULE_PATH)
backend = importlib.util.module_from_spec(spec)
spec.loader.exec_module(backend)

print('Notebook backend import resolved successfully.')
print(f'Project root: {PROJECT_ROOT}')
print(f'Backend module: {MODULE_PATH}')


## Reproducibility, runtime, and configured variants

This notebook uses the script backend as the single source of truth. The next cell runs the full toy study once, logs basic environment information, prints the exact default configuration, and lists the variant identities.


In [ ]:
start_time = time.perf_counter()
result = backend.run_experiment(verbose=False)
runtime_seconds = time.perf_counter() - start_time

config = result['config']
variant_specs = result['variant_specs']
variant_order = [spec['key'] for spec in variant_specs]
aggregates = result['aggregates']
tests = result['tests']
num_seeds = len(result['seeds'])
bootstrap_level = int(round(config['bootstrap_confidence_level']))

print('=== Reproducibility / runtime ===')
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Backend runtime for run_experiment(): {runtime_seconds:.2f} s")
print(f"Experiment id: {result['experiment_id']}")
print()

print('=== Configuration ===')
config_order = [
    'dim', 'num_layers', 'data_points', 'num_steps', 'lr', 'momentum',
    'gamma', 'scale_min', 'scale_max', 'fixed_scale_control',
    'num_seeds', 'seed_base', 'seed_stride',
    'bootstrap_resamples', 'bootstrap_confidence_level', 'bootstrap_seed'
]
for key in config_order:
    print(f'  {key:>28s}: {config[key]}')
print(f"  {'seeds':>28s}: {result['seeds']}")
print()

print('=== Variant identities ===')
for spec in variant_specs:
    print(f"  {spec['label']}: {spec['description']}")
print()
print('Scope note:')
print(f"  {result['scope_note']}")


## Final-loss summary and paired comparisons

The experiment is paired across seeds: every variant sees the same toy task and the same initialization seed offset. So the most serious second-pass summaries are:

- seedwise final losses by variant
- mean / standard deviation / standard error across seeds
- paired deltas for the key comparisons among the core five variants `(a)`, `(b)`, `(c)`, `(e)`, and `(f)`
- bootstrap confidence intervals for the paired mean deltas
- exact sign-test p-values for the paired win/loss counts

The second-pass emphasis is the **control ladder**:

- `(b)` vs `(a)`: reproduce the plain `k=20` degradation
- `(c)` vs `(b)`: show the dynamic rule rescues `k=20`
- `(f)` vs `(b)`: show whether simple fixed damping already rescues `k=20`
- `(c)` vs `(f)`: test whether the dynamic rule adds something beyond that fixed damping baseline
- `(c)` vs `(e)`: check whether the winning dynamic `k=20` result is actually stronger than the dynamic `k=5` counterpart


In [ ]:
def print_text_table(lines):
    print('\n'.join(lines))

summary_lines = []
summary_lines.append('=== Final-loss summary ===')
summary_lines.append(
    f"{'Variant':<35s} {'Mean':>12s} {'Std':>12s} {'SEM':>12s} {'Median':>12s} {'MinClamp%':>11s} {'MaxClamp%':>11s}"
)
summary_lines.append(
    f"{'-' * 35} {'-' * 12} {'-' * 12} {'-' * 12} {'-' * 12} {'-' * 11} {'-' * 11}"
)
for key in variant_order:
    agg = aggregates[key]
    sem = agg['std_final_loss'] / np.sqrt(num_seeds)
    summary_lines.append(
        f"{agg['label']:<35s} "
        f"{agg['mean_final_loss']:12.3e} {agg['std_final_loss']:12.3e} {sem:12.3e} "
        f"{agg['median_final_loss']:12.3e} {100 * agg['overall_min_clamp_fraction']:11.1f} {100 * agg['overall_max_clamp_fraction']:11.1f}"
    )
print_text_table(summary_lines)
print()

paired_lines = []
paired_lines.append('=== Core five paired final-loss comparisons (lhs - rhs) ===')
paired_lines.append(
    f"{'Comparison':<44s} {'Mean delta':>12s} {f'{bootstrap_level}% CI':>27s} {'Sign p':>9s} {'P(Δ<0)':>8s} {'LHS':>5s} {'RHS':>5s} {'Tie':>5s}"
)
paired_lines.append(
    f"{'-' * 44} {'-' * 12} {'-' * 27} {'-' * 9} {'-' * 8} {'-' * 5} {'-' * 5} {'-' * 5}"
)
for plan_item in result['core_pairwise_plan']:
    comp = tests['paired_comparisons'][plan_item['key']]
    ci = comp['bootstrap_mean_delta_ci']
    ci_str = f"[{ci['lower']:.3e}, {ci['upper']:.3e}]"
    paired_lines.append(
        f"{plan_item['headline']:<44s} {comp['mean_delta']:12.3e} {ci_str:>27s} {backend.format_pvalue(comp['exact_sign_test_pvalue']):>9s} "
        f"{ci['prob_mean_lt_zero']:8.3f} {comp['lhs_wins']:5d} {comp['rhs_wins']:5d} {comp['ties']:5d}"
    )
print_text_table(paired_lines)
print()

momentum_comp = tests['paired_comparisons']['d_minus_b']
momentum_ci = momentum_comp['bootstrap_mean_delta_ci']
print('=== Additional momentum failure check ===')
print(
    f"{momentum_comp['headline']}: mean delta={momentum_comp['mean_delta']:.3e}, "
    f"{bootstrap_level}% CI=[{momentum_ci['lower']:.3e}, {momentum_ci['upper']:.3e}], "
    f"sign-test p={backend.format_pvalue(momentum_comp['exact_sign_test_pvalue'])}, "
    f"P(mean<0)={momentum_ci['prob_mean_lt_zero']:.3f}, wins={momentum_comp['lhs_wins']}/{momentum_comp['non_ties']}"
)
print()

per_seed_lines = []
per_seed_lines.append('=== Per-seed final losses ===')
header = f"{'Seed':>6s}"
for key in variant_order:
    header += f"  {aggregates[key]['short_label']:>12s}"
header += f"  {'Best':>8s}"
per_seed_lines.append(header)
per_seed_lines.append(f"{'-' * 6}" + ''.join([f"  {'-' * 12}" for _ in variant_order]) + f"  {'-' * 8}")
for row_index, seed in enumerate(result['seeds']):
    row = f"{seed:6d}"
    row_losses = {}
    for key in variant_order:
        value = aggregates[key]['final_losses'][row_index]
        row += f"  {value:12.3e}"
        row_losses[aggregates[key]['short_label']] = value
    row += f"  {min(row_losses, key=row_losses.get):>8s}"
    per_seed_lines.append(row)
print_text_table(per_seed_lines)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(variant_order))

for idx, key in enumerate(variant_order):
    losses = aggregates[key]['final_losses']
    jitter = np.linspace(-0.12, 0.12, len(losses))
    axes[0].scatter(np.full(len(losses), idx) + jitter, losses, s=40, alpha=0.75)
    axes[0].errorbar(
        idx,
        aggregates[key]['mean_final_loss'],
        yerr=aggregates[key]['std_final_loss'],
        fmt='o',
        color='black',
        capsize=4,
        markersize=6,
        zorder=5,
    )

axes[0].set_xticks(x)
axes[0].set_xticklabels([aggregates[key]['short_label'] for key in variant_order])
axes[0].set_yscale('log')
axes[0].set_ylabel('Final loss (log scale)')
axes[0].set_title('Seedwise final losses with mean ± std')

for seed_index, seed in enumerate(result['seeds']):
    y_values = [aggregates[key]['final_losses'][seed_index] for key in variant_order]
    axes[1].plot(x, y_values, marker='o', alpha=0.35)

mean_values = [aggregates[key]['mean_final_loss'] for key in variant_order]
axes[1].plot(x, mean_values, color='black', linewidth=2.5, marker='s', label='mean over seeds')
axes[1].set_xticks(x)
axes[1].set_xticklabels([aggregates[key]['short_label'] for key in variant_order])
axes[1].set_yscale('log')
axes[1].set_ylabel('Final loss (log scale)')
axes[1].set_title('Paired seed trajectories across variants')
axes[1].legend(loc='best')

fig.suptitle('Final-loss summary by variant', fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
comp_cf = tests['paired_comparisons']['c_minus_f']
ci_cf = comp_cf['bootstrap_mean_delta_ci']
comp_fb = tests['paired_comparisons']['f_minus_b']
ci_fb = comp_fb['bootstrap_mean_delta_ci']
comp_ce = tests['paired_comparisons']['c_minus_e']
ci_ce = comp_ce['bootstrap_mean_delta_ci']

print('=== Interpretation: final-loss summary ===')
print(f"Plain k=20 worse than plain k=5? {tests['t1_k20_worse_than_k5']['pass']} | "
      f"mean(b)={aggregates['b']['mean_final_loss']:.3e} vs mean(a)={aggregates['a']['mean_final_loss']:.3e}")
print(f"Dynamic k=20 norm-ratio rescaling improves plain k=20? {tests['t2_norm_ratio_improves_k20']['pass']} | "
      f"mean(c)={aggregates['c']['mean_final_loss']:.3e}")
print(f"Fixed-scale control also improves plain k=20? {comp_fb['mean_delta'] < 0.0} | "
      f"mean(f)={aggregates['f']['mean_final_loss']:.3e}, wins={comp_fb['lhs_wins']}/{num_seeds}, "
      f"{bootstrap_level}% CI=[{ci_fb['lower']:.3e}, {ci_fb['upper']:.3e}]")
print(f"Dynamic-vs-fixed paired wins for (c): {tests['t6_norm_ratio_vs_fixed_control']['wins']}/{num_seeds} | "
      f"mean(c-f)={comp_cf['mean_delta']:.3e}, sign-test p={backend.format_pvalue(comp_cf['exact_sign_test_pvalue'])}, P(mean<0)={ci_cf['prob_mean_lt_zero']:.3f}")
print(f"Dynamic k=20 versus dynamic k=5: mean(c-e)={comp_ce['mean_delta']:.3e}, "
      f"wins for (c)={comp_ce['lhs_wins']}/{num_seeds}, {bootstrap_level}% CI=[{ci_ce['lower']:.3e}, {ci_ce['upper']:.3e}]")
print()
print('Reading guide: the first panel gives an uncertainty/seed view; the second highlights the paired design.')
print('Second-pass strengthening: the notebook now shows the control ladder explicitly, so the reader can separate')
print('plain-k20 rescue, fixed-damping rescue, and the narrower dynamic-vs-fixed residual advantage.')


## Mean loss trajectories

The next figure shows optimization trajectories, averaged over seeds, on a log-loss scale. The left panel emphasizes the stable variants; the right panel keeps the failing momentum-norm rescaling visible without compressing the rest into a single line.


In [ ]:
steps = np.arange(config['num_steps'] + 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True)

stable_keys = ['a', 'b', 'c', 'e', 'f']
for key in stable_keys:
    mean_curve = aggregates[key]['mean_loss_curve']
    std_curve = aggregates[key]['std_loss_curve']
    lower = np.maximum(mean_curve - std_curve, 1e-16)
    upper = np.maximum(mean_curve + std_curve, 1e-16)
    axes[0].plot(steps, mean_curve, linewidth=2, label=aggregates[key]['short_label'])
    axes[0].fill_between(steps, lower, upper, alpha=0.12)

axes[0].set_yscale('log')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Mean loss across seeds')
axes[0].set_title('Stable variants')
axes[0].legend(loc='best')

focus_keys = ['b', 'c', 'd', 'f']
for key in focus_keys:
    mean_curve = aggregates[key]['mean_loss_curve']
    std_curve = aggregates[key]['std_loss_curve']
    lower = np.maximum(mean_curve - std_curve, 1e-16)
    upper = np.maximum(mean_curve + std_curve, 1e-16)
    axes[1].plot(steps, mean_curve, linewidth=2, label=aggregates[key]['short_label'])
    axes[1].fill_between(steps, lower, upper, alpha=0.12)

axes[1].set_yscale('log')
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Mean loss across seeds')
axes[1].set_title('Including failed momentum-norm rescaling')
axes[1].legend(loc='best')

fig.suptitle('Loss trajectories', fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
print('=== Interpretation: trajectories ===')
for key in ['a', 'b', 'c', 'e', 'f']:
    curve = aggregates[key]['mean_loss_curve']
    print(f"{aggregates[key]['label']}: step 0={curve[0]:.3e}, step 50={curve[50]:.3e}, step 500={curve[-1]:.3e}")
print()
print('The plain k=20 baseline (b) remains worse than plain k=5 (a) by the end of training.')
print('The dynamic norm-ratio variants (c) and (e) collapse to much lower final losses.')
print('The fixed-scale control (f) is also strong, so the control comparison matters for mechanism claims.')
print('The momentum-norm variant (d) visibly fails rather than merely underperforming slightly.')


## Scale trajectories and clamp diagnostics

The script now returns both the **applied** scale trajectory and the **unclipped** scalar ratio when available, plus stepwise and overall clamp-hit fractions. This is the key reporting fix from the audit: we should not discuss the rescaling mechanism without showing whether it spends most of its time on the clamp boundary.

The caveat is stronger than a generic warning. If `(c)` and `(e)` spend nearly all steps at `scale_min = 0.1`, then the dynamic rule is only occasionally departing from the fixed-scale control. In that case, the empirical `(c)` vs `(f)` gap is still interesting, but it should be interpreted as a **small residual benefit around a clamp-floor regime**, not as evidence that explicit curvature has been restored.


In [ ]:
rescaled_keys = ['c', 'd', 'e', 'f']
scale_steps = np.arange(config['num_steps'])
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex='col')

for key in rescaled_keys:
    axes[0, 0].plot(scale_steps, aggregates[key]['mean_scale_curve'], linewidth=2, label=aggregates[key]['short_label'])
axes[0, 0].axhline(config['scale_min'], color='black', linestyle='--', linewidth=1, label='scale_min')
axes[0, 0].axhline(config['scale_max'], color='gray', linestyle=':', linewidth=1, label='scale_max')
axes[0, 0].set_ylabel('Mean applied scale')
axes[0, 0].set_title('Applied scale trajectories')
axes[0, 0].legend(loc='best')

for key in ['c', 'e']:
    axes[0, 1].plot(scale_steps, aggregates[key]['mean_unclipped_scale_curve'], linewidth=2, label=aggregates[key]['short_label'])
axes[0, 1].axhline(config['scale_min'], color='black', linestyle='--', linewidth=1, label='scale_min')
axes[0, 1].axhline(config['scale_max'], color='gray', linestyle=':', linewidth=1, label='scale_max')
axes[0, 1].set_ylabel('Mean unclipped scale')
axes[0, 1].set_title('Unclipped norm-ratio trajectories for winning variants')
axes[0, 1].legend(loc='best')

for key in ['c', 'e', 'd']:
    axes[1, 0].plot(scale_steps, 100 * aggregates[key]['mean_step_min_clamp_fraction'], linewidth=2, label=aggregates[key]['short_label'])
axes[1, 0].set_xlabel('Training step')
axes[1, 0].set_ylabel('Min-clamp occupancy (%)')
axes[1, 0].set_title('Stepwise lower-clamp occupancy')
axes[1, 0].legend(loc='best')

x = np.arange(len(rescaled_keys))
width = 0.35
min_fracs = [100 * aggregates[key]['overall_min_clamp_fraction'] for key in rescaled_keys]
max_fracs = [100 * aggregates[key]['overall_max_clamp_fraction'] for key in rescaled_keys]
axes[1, 1].bar(x - width / 2, min_fracs, width=width, label='min clamp hit %')
axes[1, 1].bar(x + width / 2, max_fracs, width=width, label='max clamp hit %')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels([aggregates[key]['short_label'] for key in rescaled_keys])
axes[1, 1].set_xlabel('Variant')
axes[1, 1].set_ylabel('Fraction of layer-steps (%)')
axes[1, 1].set_title('Overall clamp-hit fractions')
axes[1, 1].legend(loc='best')

fig.suptitle('Scale diagnostics', fontsize=14)
fig.tight_layout()
plt.show()

clamp_lines = []
clamp_lines.append('=== Clamp diagnostics ===')
clamp_lines.append(
    f"{'Variant':<35s} {'step0 scale':>12s} {'step499 scale':>14s} {'step0 unclipped':>16s} {'step499 unclipped':>18s} {'MinClamp%':>11s} {'MaxClamp%':>11s}"
)
clamp_lines.append(
    f"{'-' * 35} {'-' * 12} {'-' * 14} {'-' * 16} {'-' * 18} {'-' * 11} {'-' * 11}"
)
for key in rescaled_keys:
    agg = aggregates[key]
    clamp_lines.append(
        f"{agg['label']:<35s} {agg['mean_scale_curve'][0]:12.3f} {agg['mean_scale_curve'][-1]:14.3f} "
        f"{agg['mean_unclipped_scale_curve'][0]:16.3f} {agg['mean_unclipped_scale_curve'][-1]:18.3f} "
        f"{100 * agg['overall_min_clamp_fraction']:11.1f} {100 * agg['overall_max_clamp_fraction']:11.1f}"
    )
print('\n'.join(clamp_lines))


In [ ]:
print('=== Interpretation: clamp dominance ===')
for key in ['c', 'e']:
    frac = 100 * aggregates[key]['overall_min_clamp_fraction']
    print(f"{aggregates[key]['label']} hits the lower clamp on {frac:.1f}% of layer-steps.")
print(f"{aggregates['d']['label']} hits the upper clamp on {100 * aggregates['d']['overall_max_clamp_fraction']:.1f}% of layer-steps.")
print()
print('This is the main mechanistic caveat of the pair.')
print('The successful dynamic norm-ratio variants spend most layer-steps at the minimum clamp,')
print('so the dynamic rule is usually behaving like the fixed 0.1 control with only intermittent departures.')
print('Therefore the c-vs-f result should be read as evidence for a residual empirical advantage within a clamp-floor regime,')
print('not as evidence that this toy study has restored or measured curvature in any explicit sense.')


## Fixed-scale control versus dynamic norm-ratio rule

This is the most useful first-pass control from the audit. If the dynamic rule `(c)` and the fixed-scale control `(f)` are nearly indistinguishable, the evidence is consistent with a simple damping story. If `(c)` is materially better, then the norm-ratio heuristic may provide something beyond constant scaling in this toy setting.

The second-pass addition here is modest but useful: report the **paired delta**, a **bootstrap confidence interval** for the mean paired delta, and an **exact sign-test p-value** for the paired win/loss pattern. The notebook also now reminds the reader that `(f)` versus `(b)` matters: if fixed damping alone already rescues `k=20`, then the incremental claim for `(c)` should be limited to whether it improves on that already-strong fixed baseline.


In [ ]:
losses_c = aggregates['c']['final_losses']
losses_f = aggregates['f']['final_losses']
comp_cf = tests['paired_comparisons']['c_minus_f']
comp_fb = tests['paired_comparisons']['f_minus_b']
comp_ce = tests['paired_comparisons']['c_minus_e']
control_test = tests['t6_norm_ratio_vs_fixed_control']
ci_cf = comp_cf['bootstrap_mean_delta_ci']
ci_fb = comp_fb['bootstrap_mean_delta_ci']
ci_ce = comp_ce['bootstrap_mean_delta_ci']
delta_cf = comp_cf['delta_values']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for index, seed in enumerate(result['seeds']):
    color = 'tab:green' if losses_c[index] < losses_f[index] else 'tab:red'
    axes[0].plot([0, 1], [losses_c[index], losses_f[index]], marker='o', color=color, alpha=0.7)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels([aggregates['c']['short_label'], aggregates['f']['short_label']])
axes[0].set_yscale('log')
axes[0].set_ylabel('Final loss (log scale)')
axes[0].set_title('Paired seed comparison: dynamic vs fixed')

axes[1].scatter(losses_f, losses_c, s=60, alpha=0.8)
min_val = min(losses_c.min(), losses_f.min())
max_val = max(losses_c.max(), losses_f.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel(f"{aggregates['f']['short_label']} final loss")
axes[1].set_ylabel(f"{aggregates['c']['short_label']} final loss")
axes[1].set_title('Diagonal = equal performance')

core_comp_keys = ['c_minus_b', 'f_minus_b', 'c_minus_f', 'c_minus_e']
core_comp_labels = [tests['paired_comparisons'][key]['headline'] for key in core_comp_keys]
core_comp_means = [tests['paired_comparisons'][key]['mean_delta'] for key in core_comp_keys]
core_comp_lowers = [tests['paired_comparisons'][key]['bootstrap_mean_delta_ci']['lower'] for key in core_comp_keys]
core_comp_uppers = [tests['paired_comparisons'][key]['bootstrap_mean_delta_ci']['upper'] for key in core_comp_keys]
centers = np.arange(len(core_comp_keys))
errors = np.vstack([
    np.array(core_comp_means) - np.array(core_comp_lowers),
    np.array(core_comp_uppers) - np.array(core_comp_means),
])
axes[2].errorbar(core_comp_means, centers, xerr=errors, fmt='o', color='black', capsize=4)
axes[2].axvline(0.0, color='gray', linestyle='--', linewidth=1)
axes[2].set_yticks(centers)
axes[2].set_yticklabels(core_comp_labels)
axes[2].set_xlabel('Paired mean delta (lhs - rhs)')
axes[2].set_title(f'Key paired deltas with {bootstrap_level}% bootstrap CI')

fig.suptitle('Dynamic norm-ratio rule vs fixed-scale control', fontsize=14)
fig.tight_layout()
plt.show()

print('=== Control comparison summary ===')
print(f"mean(c)={aggregates['c']['mean_final_loss']:.3e}")
print(f"mean(f)={aggregates['f']['mean_final_loss']:.3e}")
print(f"mean(c - f)={comp_cf['mean_delta']:.3e} with std={comp_cf['std_delta']:.3e}")
print(f"median(c - f)={comp_cf['median_delta']:.3e}")
print(f"paired wins for (c) over (f): {comp_cf['lhs_wins']}/{num_seeds} (rhs wins={comp_cf['rhs_wins']}, ties={comp_cf['ties']})")
print(f"relative difference vs fixed control: {control_test['pct_vs_fixed']:+.1f}%")
print(f"{bootstrap_level}% bootstrap CI for mean(c - f): [{ci_cf['lower']:.3e}, {ci_cf['upper']:.3e}]")
print(f"CI excludes zero? {ci_cf['excludes_zero']}")
print(f"exact sign-test p={backend.format_pvalue(comp_cf['exact_sign_test_pvalue'])} over {comp_cf['non_ties']} non-tied seeds")
print(f"bootstrap P(mean(c - f) < 0)={ci_cf['prob_mean_lt_zero']:.3f}")
print(f"fixed control vs plain k=20: mean(f - b)={comp_fb['mean_delta']:.3e}, {bootstrap_level}% CI=[{ci_fb['lower']:.3e}, {ci_fb['upper']:.3e}], wins={comp_fb['lhs_wins']}/{num_seeds}")
print(f"dynamic k=20 vs dynamic k=5: mean(c - e)={comp_ce['mean_delta']:.3e}, {bootstrap_level}% CI=[{ci_ce['lower']:.3e}, {ci_ce['upper']:.3e}], wins for (c)={comp_ce['lhs_wins']}/{num_seeds}")
print(f"similar within 10%? {control_test['similar_within_10pct']}")
print(f"seedwise deltas (c - f): {np.array2string(delta_cf, precision=2, suppress_small=False)}")


## Calibrated conclusion

The notebook should end by separating **what was shown empirically** from **what was not shown mechanistically**.


In [ ]:
print('=== Calibrated conclusion ===')
for line in result['verdict_lines']:
    print(f'- {line}')
print()
print('Second-pass strengthening summary:')
print('- the script and notebook now expose a shared core-pairwise plan for the main (a,b,c,e,f) comparisons;')
print('- the evidence ladder now distinguishes plain-k20 degradation, fixed-damping rescue, and dynamic-vs-fixed residual gain;')
print('- clamp dominance is now stated more explicitly as a clamp-floor caveat rather than a generic warning.')
print()
print('Current empirical takeaway: the dynamic norm-ratio k=20 rule is still the best mean performer in this toy setup,')
print('and it beats the fixed 0.1 control on most seeds; however, because the winning dynamic variants sit near the lower clamp most of the time,')
print('the mechanism claim should stay narrow: this pair shows an empirical optimization advantage for a clipped scalar heuristic, not curvature restoration.')
